In [1]:
import sys
sys.path.append("..")   # let the notebook find src/

import numpy as np
import pandas as pd

from src.classifier import IntentClassifier, load_abcd_data
from src.env import Handoffenv, build_scenarios, is_bot_solvable, ANSWER, CLARIFY, HANDOFF
from src.baseline import always_handoff, never_handoff, confidence_handoff, classifier_handoff

In [2]:
data = load_abcd_data("../data/abcd_v1.1.json.gz")
intent_clf = IntentClassifier.load("../models/intent_classifier.joblib")
solvable_clf = IntentClassifier.load("../models/solvable_classifier.joblib")

print({split: len(convos) for split, convos in data.items()})       # expect 8034 / 1004 / 1004
print(intent_clf.model.classes_[:3], "...")                         # intent names
print(solvable_clf.model.classes_)                                  #[False  True]

{'train': 8034, 'dev': 1004, 'test': 1004}
['account_access' 'manage_account' 'order_issue'] ...
[False  True]


In [3]:
scenarios = build_scenarios(data["dev"], intent_clf)

dev_env_balanced = Handoffenv(scenarios, intent_clf.model.classes_, seed = 0, solvable_ratio = 0.5)
dev_env_natural  = Handoffenv(scenarios, intent_clf.model.classes_, seed = 0, solvable_ratio = None)

print("dev scenarios:", len(dev_env_balanced.scenarios))
print("bot-solvable:", len(dev_env_balanced.solvable),
      "| needs human:", len(dev_env_balanced.unsolvable))

obs, _ = dev_env_balanced.reset(seed=3)
dev_env_balanced.render()

dev scenarios: 1004
bot-solvable: 185 | needs human: 819
[turn 0] customer: 'Hello' | bot thinks: storewide_query (0.14) | frustration=0 | asked_human=False


In [4]:
def evaluate(policy, env, n_episodes=500):
    """Run a policy for n_episodes chats and summarize the total reward per chat.

    policy: a function that takes env and returns an action (0, 1, or 2)
    """
    rewards = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=i)         
        done = False
        total_reward = 0.0
        while not done:
            action = policy(env)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        rewards.append(total_reward)

    rewards = np.array(rewards)
    return {"mean": rewards.mean(), "std": rewards.std(),
            "min": rewards.min(), "max": rewards.max()}

In [5]:
policies = {
    "always_handoff": always_handoff,
    "never_handoff": lambda env:  never_handoff(env, n =1),
    "confidence":     confidence_handoff,
    "classifier":     lambda env: classifier_handoff(env, solvable_clf),
}

for env_name, env in [("balanced", dev_env_balanced), ("natural", dev_env_natural)]:
    rows = [{"policy": name, **evaluate(policy, env)} for name, policy in policies.items()]
    print(env_name)
    display(pd.DataFrame(rows).round(2))

balanced


,policy,mean,std,min,max
0,always_handoff,-3.00,0.00,-3.0,-3.0
1,never_handoff,-21.69,24.72,-78.0,8.0
2,confidence,-14.90,21.87,-67.0,9.0
3,classifier,-0.51,7.75,-61.0,9.0


natural


,policy,mean,std,min,max
0,always_handoff,-3.00,0.00,-3.0,-3.0
1,never_handoff,-32.88,21.57,-62.0,8.0
2,confidence,-24.12,19.75,-67.0,9.0
3,classifier,-2.76,6.57,-67.0,9.0
